# NCS inference (GPU): run once per volume and model

Runs one of the NCS seismic foundation models (Norsk Regnesentral; Ordoñez et al. 2025, arXiv 2603.23211; https://github.com/NorskRegnesentral/NCS_models) on the seismic crop and saves the 768-channel feature cube to Drive. **Analysis happens in `RGT-Est_NCS_Analysis_Colab.ipynb`**, which runs on a CPU runtime.

| `MODEL_KIND` | What it looks at | Default batch |
|---|---|---|
| `"2d"` | single sections (inlines with `dir0`) | 64 |
| `"2.5d"` | several 2D views through each point (the model config's views, or `INPUT_VIEWS`) | 32 |
| `"3d"` | true 3D crops | 4 |

- **Weights** are downloaded once per model into `<DRIVE_FOLDER>/models/` and loaded from there afterwards.
- **The cube** is saved as `<DRIVE_FOLDER>/outputs/<seismic>_ncs_<model>_<direction>_dens<D>.zarr`, with a `.json` record of the settings, GPU and model version. If it already exists, inference is skipped.

**Runtime:** **A100 GPU** recommended for 3D (Runtime → Change runtime type). Then **Runtime → Run all**.

## 1. Settings

In [ ]:
DATA_SOURCE  = "drive"
DRIVE_FOLDER = "/content/drive/MyDrive/PUBLIC"
DATA_FOLDER  = "/content"
SEGY_FILE    = "Full_Stack_8_41_CROP_1793889427461.sgy"

MODEL_KIND   = "3d"              # "2d", "2.5d" or "3d"
DIRECTION    = "dir0"            # primary traversal: "dir0" (inline) or "dir90" (crossline)
INPUT_VIEWS  = None              # 2.5D only: e.g. ["dir0", "dir90"]; None = the model's own views
DENSIFY      = 4                 # embedding every 16/DENSIFY samples (4 → every 4; 8 → every 2, ~8x bigger)
DTYPE        = "float16"         # "float16" or "bfloat16" on GPU
BATCH        = None              # None = default for the model (64 / 32 / 4)
NUM_OVERLAP  = 7

## 2. Set up

In [ ]:
!pip -q install segyio "NCS[inference] @ git+https://github.com/NorskRegnesentral/NCS_models"
import os, time, shutil, json
import numpy as np, torch, segyio
MODELS = {"2d": ("NorskRegnesentralSTI/NCS-v1-2d-base", 64), "2.5d": ("NorskRegnesentralSTI/NCS-v1-2.5d-base", 32),
          "3d": ("NorskRegnesentralSTI/NCS-v1-3d-base", 4)}
MODEL_ID, default_batch = MODELS[MODEL_KIND]
BATCH = BATCH or default_batch
TAG = MODEL_KIND.replace(".", "")
if not torch.cuda.is_available():
    raise RuntimeError("No GPU: Runtime → Change runtime type → A100 (or L4) GPU.")
GPU = torch.cuda.get_device_name(0)
print(f"PyTorch {torch.__version__} | GPU {GPU} ({torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB) | model {MODEL_ID} | batch {BATCH}")

if DATA_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    src, dst = os.path.join(DRIVE_FOLDER, SEGY_FILE), os.path.join(DATA_FOLDER, SEGY_FILE)
    if not (os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src)):
        shutil.copyfile(src, dst); print("copied", SEGY_FILE)
SEGY_PATH = os.path.join(DATA_FOLDER, SEGY_FILE)
BASE = DRIVE_FOLDER if DATA_SOURCE == "drive" else DATA_FOLDER
stem = os.path.splitext(SEGY_FILE)[0]
with segyio.open(SEGY_PATH) as f:                       # geometry the way NCS reads it (bytes 189/193)
    print(f"seismic as NCS sees it: {len(f.ilines)} inlines × {len(f.xlines)} crosslines × {len(f.samples)} samples")

## 3. Model weights: downloaded once into Drive

If Colab asks to grant access to a secret (`HF_TOKEN`), "Don't grant" is fine: the models are public.

In [ ]:
MODEL_DIR = os.path.join(BASE, "models", MODEL_ID.split("/")[-1])
if os.path.exists(os.path.join(MODEL_DIR, "config.json")):
    print("model weights from Drive:", MODEL_DIR)
else:
    from huggingface_hub import snapshot_download, HfApi
    tmp_model = "/content/_ncs_model_download"
    if os.path.exists(tmp_model): shutil.rmtree(tmp_model)
    t0 = time.time()
    snapshot_download(repo_id=MODEL_ID, local_dir=tmp_model)
    try:
        sha = HfApi().model_info(MODEL_ID).sha
    except Exception as e:
        sha = f"unknown ({type(e).__name__})"
    with open(os.path.join(tmp_model, "_source.txt"), "w") as fh:
        fh.write(f"source: https://huggingface.co/{MODEL_ID}\ncommit: {sha}\n"
                 f"downloaded (UTC): {time.strftime('%Y-%m-%d %H:%M', time.gmtime())}\n")
    for junk in (".cache", ".gitattributes"):
        jp = os.path.join(tmp_model, junk)
        if os.path.isdir(jp): shutil.rmtree(jp)
        elif os.path.isfile(jp): os.remove(jp)
    os.makedirs(os.path.dirname(MODEL_DIR), exist_ok=True)
    shutil.copytree(tmp_model, MODEL_DIR, dirs_exist_ok=True)
    print(f"downloaded {MODEL_ID} in {time.time()-t0:.0f} s → {MODEL_DIR}")
print(open(os.path.join(MODEL_DIR, "_source.txt")).read() if os.path.exists(os.path.join(MODEL_DIR, "_source.txt")) else "")

## 4. Run inference (skipped if the cube is already in Drive)

In [ ]:
FEAT_PATH = os.path.join(BASE, "outputs", f"{stem}_ncs_{TAG}_{DIRECTION}_dens{DENSIFY}.zarr")
LEGACY = os.path.join(BASE, "outputs", f"{stem}_ncs_{DIRECTION}_dens{DENSIFY}.zarr")   # first 2D run's name
if MODEL_KIND == "2d" and not os.path.exists(FEAT_PATH) and os.path.exists(LEGACY):
    FEAT_PATH = LEGACY
if os.path.exists(FEAT_PATH):
    print("feature cube already in Drive:", FEAT_PATH)
else:
    from NCS.inference import run_inference
    tmp = f"/content/{os.path.basename(FEAT_PATH)}"
    if os.path.exists(tmp): shutil.rmtree(tmp)
    kw = dict(model_path=MODEL_DIR, input_path=SEGY_PATH, output_path=tmp, direction=DIRECTION, densify=DENSIFY,
              num_overlap_patches=NUM_OVERLAP, overlap_filter="ramp", batch_size=BATCH, device="cuda", dtype=DTYPE)
    if INPUT_VIEWS: kw["input_views"] = list(INPUT_VIEWS)
    t0 = time.time()
    run_inference(**kw)
    minutes = (time.time() - t0) / 60
    print(f"inference took {minutes:.1f} min on {GPU}")
    shutil.copytree(tmp, FEAT_PATH)
    record = dict(model=MODEL_ID, model_dir=MODEL_DIR, segy=SEGY_FILE, direction=DIRECTION, input_views=INPUT_VIEWS,
                  densify=DENSIFY, dtype=DTYPE, batch=BATCH, num_overlap=NUM_OVERLAP, gpu=GPU, minutes=round(minutes, 1),
                  utc=time.strftime("%Y-%m-%d %H:%M", time.gmtime()))
    with open(FEAT_PATH + ".json", "w") as fh: json.dump(record, fh, indent=1)
    print("saved to", FEAT_PATH)

## 5. Quick check

In [ ]:
import xarray as xr
groups = sorted(d for d in os.listdir(FEAT_PATH) if os.path.isdir(os.path.join(FEAT_PATH, d)))
ds = xr.open_zarr(FEAT_PATH, group=DIRECTION if DIRECTION in groups else groups[0])
print("groups:", groups); print(ds)
print("\nNext: open RGT-Est_NCS_Analysis_Colab.ipynb (a CPU runtime is enough).")